## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161397?b2cUser=true"><b>Langchain Agentes - Revisando o prompt e analisando o não-determinismo</b></a><br/>

<b>Objetivo:</b> Criação de uma nova ferramenta chamada Perfil Acadêmico que recebe os dados da primeira.<br/>

<b>PASSOS:</b><br/>
<ul>
    <b><li>CRIAÇÃO DAS FERRAMENTAS</li></b><br/>
    <ul>
        <ol>
            <li>Criação da nova Ferramenta PerfilAcadêmico</li>
            <ul><li>Informar na sua descrição que NÃO consegue obter os dados do estudante sozinha e que eles devem ser buscados antes</li></ul>
            <li>Instanciando as Ferramentas que a LLM precisa usar</li><br/>
            <ul>
                <li><b>return_direct=False</b> -> O AGENTE PASSA PELA FASE DE RACIOCÍNIO ANTES DE RETORNAR A RESPOSTA. USADO NA FERRAMENTA INTERMEDIÁRIA.</li><br/>
                <li><b>return_direct=True (Default)</b> -> O AGENTE RETORNA DIRETAMENTE A RESPOSTA, SEM PASSAR PELO RACIÓCINIO DO AGENTE. USADO NA FERRAMENTA FINAL</li>     
            </ul>             
        </ol>
    </ul><br/>
    <b><li>CRIAÇÃO DO AGENTE DE FERRAMENTAS</li></b><br/>   
    <ul>
        <ol>
            <li>Informando para a LLM as ferramentas que eu tenho (Usa as ferramentas que foram instanciadas)</li>
            <li>Executando o Agente com as ferramentas</li>
        </ol>
    </ul><br/>
</ul>

In [1]:
#%pip install -r requirements.txt

In [2]:
from pydantic import BaseModel, Field
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pandas import read_csv

#from langchain.globals import set_debug

#set_debug(True)

class ExtratordeEstudante(BaseModel):
    estudante: str = Field(description="Nome do estudante informado, sempre em letras minúsculas. Exemplo: joão, carla, joana")

# FERRAMENTA DADOS DE ESTUDANTE
class DadosDeEstudante(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                        """ # Descrição da ferramenta    
    
    def __init__(self,llm:ChatOpenAI):
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        self.llm = llm  
        print('Inicializando ferramenta Dados de Estudante')      
    
    def __busca_dados_de_estudante(self,estudante:str) -> str:
        
        dfestudantes = read_csv("documentos/estudantes.csv")
        
        dados_estudante = dfestudantes.loc[dfestudantes['USUARIO'] == estudante]
        
        if dados_estudante.empty:
            return f"Desculpe, não encontrei dados para o estudante '{estudante}'. Por favor, verifique o nome e tente novamente."
        
        #print(estudante)
        
        dict = dados_estudante.to_dict(orient='records')[0]
        
        return dict
    
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:        
        
        parseador = JsonOutputParser(pydantic_object=ExtratordeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    Você deve analisar a {input} e extrair o nome de estudante informado.
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                """,
                                    input_variables = ["input"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador
        
        resposta = cadeia.invoke({"input": input})
        
        estudante = resposta['estudante']
        
        print('Retorno estudante da LLM:', estudante)        
        
        dados = self.__busca_dados_de_estudante(estudante)
        print('Dicionário retornado pelo método busca_dados_de_estudante:', dados)
               
        return dados


In [3]:
from typing import List

class Nota(BaseModel):
    area_de_conhecimento: str = Field(description="Área de conhecimento da nota")
    nota: float = Field(description="Nota obtida pelo estudante nessa área de conhecimento")
    
class ExtratorPerfilAcademicoDeEstudante(BaseModel):
    
    nome:str = Field(description="Nome do estudante")
    ano_de_conclusao: int = Field(description="Ano de conclusão")
    notas: List[Nota] = Field(description="Lista de notas para cada área de conhecimento") # O Nota dentro da lista, é a classe criada acima, que 
                                                                                           # formata o dicionário de notas
                                                                                           
    resumo: str = Field(description="Resumo das principais características desse estudante de forma a torná-lo único e um ótimo potencial estudante para faculdades. Só esse estudante tem bla bla bla")

### <b>CRIAÇÃO DE FERRAMENTAS</b>
Que ferramentas eu tenho disponíveis para se obter os dados da Ana ?

<b>1) Criação da Ferramenta Perfil Acadêmico</b>
<ul><li>Informar na sua descrição que NÃO consegue obter os dados do estudante sozinha e que eles devem ser buscados antes</li></ul>


In [4]:
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI


class PerfilAcademico(BaseTool): # ESTENDE BaseTool
    
    llm:ChatOpenAI = None
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "perfil_academico" # Nome da ferramenta
    description : str = """                                                       
                            - Esta ferramenta utiliza os dados do estudante para gerar um perfil acadêmico detalhado.
                            
                            - Eu sou incapaz de obter os dados do estudante sozinha. Buscar os dados do estudante antes de me invocar.                            
                        """ # Descrição da ferramenta    
    
    def __init__(self,llm:ChatOpenAI):
        super().__init__() # PARA NÃO SOBRESCREVER O CONSTRUTOR DA CLASSE MÃE BaseTool
        self.llm = llm   
        print('Inicializando ferramenta Perfil Acadêmico')
        
                
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    #
    # JÁ QUE SE TRATA DA ENTRADA DOS DADOS DE UM ESTUDANTE NO FORMATO TEXTO, SERÁ UTILIZADA A SAÍDA DA FERRAMENTA DE DADOS DE ESTUDANTE 
    # COMO ENTRADA DESTA FERRAMENTA DE PERFIL ACADÊMICO.
    #
    # ASSIM, A ENTRADA DESTE MÉTODO _run SERÁ O TEXTO COM OS DADOS DO ESTUDANTE, COMO CONTEXTO DE UM PROMPT, QUE SERÁ USADO 
    # PARA INFORMAR A MANEIRA COMO O PERFIL ACADÊMICO DEVE SER GERADO.
    # 
    def _run(self, input: str) -> str:        
        
        print("Entrada Perfil Acadêmico\n", input)
        
        parseador = JsonOutputParser(pydantic_object=ExtratorPerfilAcademicoDeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    CONTEXTO:                            
                                                    Você é uma consultora de carreiras
                                                                                
                                                    Esta ferramenta utiliza os dados do estudante para gerar um perfil acadêmico detalhado.
                                                    
                                                    OBJETIVO:
                                                        - Criar o perfil acadêmico de um estudante utilizando os dados fornecidos {dados_do_estudante}
                                                    
                                                    ESTILO:
                                                    Precisa indicar com detalhes, riqueza, mas direta ao ponto.
                                                                                
                                                    PASSOS:
                                                        - Formate o estudante para o seu perfil acadêmico.
                                                        - Com os dados, identifique as opções de universidades sugeridas e cursos compatíveis com o interesse do aluno.
                                                        - Destaque o perfil do aluno, dando ênfase, principalmente, naquilo que faz interesse nas instituições de interesse
                                                        do aluno.                                                      
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                """,
                                    input_variables = ["dados_do_estudante"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | self.llm | parseador        
        
        resposta = cadeia.invoke({"dados_do_estudante": input})
        
        print('Resposta Perfil Acadêmico\n', resposta)
        
        return resposta


<b>2) Instanciando as Ferramentas que a LLM precisa usar</b>
<ul>
    <li> Para o objeto da classe Tool, deverão ser informados o nome, a função, que é o método run da ferramenta que foi criada, e a descrição</li>
    <li> Incluíndo a segunda ferramenta</li><br/>
    <ul>
        <li><b>return_direct=False</b> -> O AGENTE PASSA PELA FASE DE RACIOCÍNIO ANTES DE RETORNAR A RESPOSTA. USADO NA FERRAMENTA INTERMEDIÁRIA.</li>
        <li><b>return_direct=True (Default)</b> -> O AGENTE RETORNA DIRETAMENTE A RESPOSTA, SEM PASSAR PELO RACIÓCINIO DO AGENTE. USADO NA FERRAMENTA FINAL</li>     
    </ul>            
</ul>

In [5]:
from langchain.agents import Tool

class Tools:
        
        def __init__(self,llm:ChatOpenAI):
                
                dados_de_estudante = DadosDeEstudante(llm) # INSTANCIANDO O OBJETO DA MINHA FERRAMENTA
                perfil_academico = PerfilAcademico(llm) # INSTANCIANDO O OBJETO DA MINHA FERRAMENTA

                # MATRIZ DE FERRAMENTAS (CONJUNTO DE FERRAMENTAS)
                self.tools = [
                        # Instanciando ferramentas
                        Tool(
                                name=dados_de_estudante.name,
                                func=dados_de_estudante.run,
                                description=dados_de_estudante.description,
                                return_direct=False # SE FALSE -> O AGENTE PASSA PELA FASE DE RACIOCÍNIO ANTES DE RETORNAR A RESPOSTA. USADO NA FERRAMENTA INTERMEDIÁRIA
                                                    # SE TRUE -> O AGENTE RETORNA DIRETAMENTE A RESPOSTA DA FERRAMENTA PARA O USUÁRIO, SEM PASSAR PELO RACIÓCINIO DO AGENTE.                                                                                            
                        ),
                        
                        # SEGUNDA FERRAMENTA. 
                        # SE VIRA PARA PEGAR OS DADOS DO ESTUDANTE E GERAR O PERFIL ACADÊMICO
                        Tool(
                                name=perfil_academico.name,
                                func=perfil_academico.run,
                                description=perfil_academico.description
                                                      
                        )
                ]

### <b>CRIAÇÃO DO AGENTE DE FERRAMENTAS</b>

<b>3) Informando para a LLM as ferramentas que eu tenho</b> 
<ul><li>Para isso, é necessário criar um agente com as ferramentas</li></ul>

In [6]:
from langchain.agents import create_openai_tools_agent
from langchain import hub
from dotenv import load_dotenv
from os import getenv
import warnings

warnings.filterwarnings("ignore")

class AgenteOpenAIFunctions:
    
    def __init__(self):
      
      load_dotenv()

      llm = ChatOpenAI(
                        model="gpt-5-mini",
                        api_key=getenv("API_KEY") 
                      )
      
      # INSTANCIANDO AS FERRAMENTAS
      self.tools = Tools(llm).tools
        
      # PROMPT DE INICIALIZAÇÃO PARA INFORMAR PARA A LLM SOBRE A FERRAMENTA.
      prompt=(hub.pull(owner_repo_commit="hwchase17/openai-functions-agent"))

      # CRIANDO UM AGENTE COM AS FERRAMENTAS
      self.agente = create_openai_tools_agent(
                                                llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                                tools=self.tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                                prompt=prompt  
                                                                  # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                                          # Para agente de função (https://smith.langchain.com/hub/hwchase17/openai-functions-agent)
                                                                                
                                             )

      print(prompt)


<b>4) Executando o agente com a ferramenta</b>

Pode-se até passar dois nomes de uma vez.

In [7]:
from langchain.agents import AgentExecutor

agente = AgenteOpenAIFunctions()

executor = AgentExecutor(
                            agent=agente.agente, # O AGENTE QUE VAI SER USADO
                            tools=agente.tools, # FERRAMENTAS QUE O AGENTE PODE USAR
                            verbose=True
                        )

for pergunta in [
                    "Crie um perfil acadêmico para a Ana.", # NECESSÁRIO RECEBER OS DADOS DA PRIMEIRA FERRAMENTA, DADOS DE ESTUDANTE, COMO ENTRADA.                    
                    "Compare o perfil acadêmico da Ana com o da Bianca."
                ]:
    
    print('\nPergunta: ', pergunta,"\n")

    resposta = executor.invoke({"input": pergunta})
    print(resposta)
    
    

Inicializando ferramenta Dados de Estudante
Inicializando ferramenta Perfil Acadêmico
input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annot